# CSCI/MATH 485 Assignment
## Customer Churn Prediction with XGBoost
## Starter Notebook

This notebook is compatible with **Jupyter Notebook** and **Google Colab**.

This starter code is only to get you started. You can change any of the existing code here to complete all the tasks.

Complete all `TODO` sections. Make sure your final submission includes:
- data analysis,
- a tuned XGBoost model,
- your chosen main evaluation metric and justification,
- interpretation of top important features,
- and a final comparison with the baseline model.


## 1. Setup


In [1]:
# If you are using Google Colab, uncomment the next line if xgboost is not installed.
!pip install xgboost



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


## 2. Load the Dataset

**Instructions for students**
- Load the IBM Telco Customer Churn dataset.
- Display the first few rows.
- Confirm the dataset shape.
- If the url doesn't work for you, download the csv file from the Canvas Assignment#4

In [3]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
df = pd.read_csv(url)

# Display the first 5 rows of the dataset (done below)
df.head()

# Display the shape of the dataset
#print(df.shape[0])     # rows
#print(df.shape[1])     # columns 

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data Exploration

Complete the following:
- Print all column names
- Show data types
- Count missing values in each column
- Show the distribution of the target variable
- Write a short note: Is this a classification or regression problem? Why is this useful in business?


In [4]:
# TODO:
# Print column names
# Print data types
# Print missing values for each column
# Print value counts for the target column

print("Columns:")
print("\nData types:")
df.info()

print("\nMissing values:")
np.sum(df.isna().values) 

print("\nTarget distribution:")
y = df['Churn'] 
print(y)


Columns:

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  70

**TODO (write your answer below):**

1. What kind of machine learning problem is this?
2. Why is churn prediction important in a business setting?

_We will be doing a classification method, to focus on Churn (turnover). This is important for businesses to help determine which contributing factors lead to overturn or costumers leaving. Time, prices and monthly bill seem to be strong variables that can affect peoples decision to stay or leave a company and focusing on these with some underlining understanding of the data (boosted method), we can gain a more accurate picture of customer outcomes._


## 4. Basic Cleaning

Complet the following:
- Identify whether there is an ID column that should be removed
- Convert the target column into binary form if needed
- Convert any numeric-looking columns stored as text into numeric values


In [5]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [6]:
# Copy
df_clean = df.copy()

# 1. Drop any unnecessary identifier column(s)
df_clean = df_clean.drop(columns=['customerID'])

# 2. Convert the target column to 0/1
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int) 

# 3. Convert any columns that should be numeric into numeric type
#df_clean['TotalCharges'] = df_clean['TotalCharges'].astype('float')

df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# 4. Handle invalid parsing if needed
# Example structure only:
# df_clean = df_clean.drop(columns=[...])
# df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
# df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

df_clean.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 5. Define Features and Target

Complet the following:
- Define `X` and `y`
- Set the correct target column


In [7]:
# TODO:
# Replace with the correct target column name
target_col = "Churn"

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (7043, 19)
Target shape: (7043,)


## 6. Identify Numeric and Categorical Features

Complet the following:
- Create a list of numeric columns
- Create a list of categorical columns


In [8]:
# TODO:
# Identify numeric and categorical feature columns

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender', 'Partner', 'Dependents', 'SeniorCitizen', 'PhoneService','MultipleLines','InternetService',
                        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                        'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
                        'PaymentMethod' ]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numeric features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'SeniorCitizen', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 7. Train/Test Split

Complet the following:
- Split the dataset into training and testing sets
- Use stratification if appropriate


In [9]:
# TODO:
# Create train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (5634, 19)
X_test shape: (1409, 19)


## 8. Preprocessing Pipelines

Build preprocessing for:
- numeric features
- categorical features

Then combine them into a `ColumnTransformer`.


In [10]:
# TODO:
from sklearn.preprocessing import StandardScaler

# Build numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'SeniorCitizen', 'PhoneService',
                                  'MultipleLines', 'InternetService',
                                  'OnlineSecurity', 'OnlineBackup',
                                  'DeviceProtection', 'TechSupport',
                                  'StreamingTV', 'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])

## 9. Baseline Model: Logistic Regression

Complet the following:
- Train a Logistic Regression model as the baseline
- Generate predictions
- Evaluate using multiple metrics
- You may need to adjust `max_iter` if the model is not converging.


In [11]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=5000))
])

# TODO:
# Fit the baseline model
# Generate predicted labels
# Generate predicted probabilities

# Fit model
baseline_model.fit(X_train, y_train)

# Predictions (labels)
baseline_preds = baseline_model.predict(X_test)

# Probabilities (for ROC-AUC)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

In [12]:
# TODO:
# Compute and print:
# - Accuracy
# - Precision
# - Recall
# - F1-score
# - ROC-AUC

# Suggested variable names:
# baseline_preds
# baseline_probs

# Accuracy
accuracy = accuracy_score(y_test, baseline_preds)
# Precision
precision = precision_score(y_test, baseline_preds)
# Recall
recall = recall_score(y_test, baseline_preds)
# F1-score
f1 = f1_score(y_test, baseline_preds)
# ROC-AUC 
roc_auc = roc_auc_score(y_test, baseline_probs)

print(f"Accuracy:  {accuracy:}")
print(f"Precision: {precision:}")
print(f"Recall:    {recall:}")
print(f"F1-score:  {f1:}")
print(f"ROC-AUC:   {roc_auc:}")


Accuracy:  0.8055358410220014
Precision: 0.6572327044025157
Recall:    0.5588235294117647
F1-score:  0.6040462427745664
ROC-AUC:   0.8418713994161565


In [13]:
# Optional but helpful
# TODO:
# Print classification report and confusion matrix

print(classification_report(y_test, baseline_preds))
print(confusion_matrix(y_test, baseline_preds))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

[[926 109]
 [165 209]]


## 10. Choose Your Main Evaluation Metric

Choose **one main metric** for this churn problem.

You must explain:
- which metric you chose,
- why it is appropriate,
- and why it is more informative than accuracy alone for this problem.


**TODO (write your answer below):**

F1- score is a great metric of determining the relationship between accuracy and percision together. Accessing both together as a ration of "when I am right" to "total ground truth positives". This is catching the information on false positives and false negatives that convey a importance due to the impact it could make in the models interpretation.   

## 11. XGBoost Model

Complet the following:
- Build an XGBoost pipeline
- Tune at least 3 hyperparameters
- Use either `GridSearchCV` or your own tuning approach


In [18]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        eval_metric="auc",
        random_state=42
    ))
])

# TODO:
# Define a hyperparameter grid with at least 3 hyperparameters
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
}

param_grid


{'classifier__n_estimators': [100, 200],
 'classifier__max_depth': [3, 5, 7],
 'classifier__learning_rate': [0.05, 0.1],
 'classifier__subsample': [0.8, 1.0]}

In [19]:
# TODO:
# Run GridSearchCV (or perform manual tuning)
# Choose a scoring metric that matches your selected main evaluation metric

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring = "roc_auc" ,   # TODO: replace with your chosen scoring metric
    cv=3,
    n_jobs=-1,
    verbose=1, 
    error_score='raise'
)

# TODO:
# Fit grid search on training data

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


GridSearchCV(cv=3, error_score='raise',
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['tenure',
                                                                          'MonthlyCharges',
                                                                          'TotalCharges']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncod...
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'classifier__learning_rate': [0.05, 0.1],
                         'classifier__max_depth': [3, 5, 7],
                         'classifier__n_estimators': [100, 200],
                         'classifier__subsample': [0.8, 1.0]},
             scoring='roc_auc', verbose=1)

In [ ]:
#print(X_train.dtypes)

In [20]:
# TODO:
# Print the best hyperparameters
# Save the best model

final_model = grid_search.best_estimator_

xgb_preds = final_model.predict(X_test)
xgb_probs = final_model.predict_proba(X_test)[:, 1]


print(grid_search.best_params_)


{'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}


## 12. Evaluate the Tuned XGBoost Model

Evaluate XGBoost using the same metrics as the baseline.


In [22]:
# TODO:
# Generate predictions and probabilities using the best XGBoost model

# Suggested variable names:
# xgb_preds
# xgb_probs

# Predictions 
xgb_preds = final_model.predict(X_test)

# Probabilities 
xgb_probs = final_model.predict_proba(X_test)[:, 1]

xgb

In [23]:
# TODO:
# Compute and print:
# - Accuracy
# - Precision
# - Recall
# - F1-score
# - ROC-AUC

xgb_accuracy = accuracy_score(y_test, xgb_preds)
xgb_precision = precision_score(y_test, xgb_preds)
xgb_recall = recall_score(y_test, xgb_preds)
xgb_f1 = f1_score(y_test, xgb_preds)
xgb_roc_auc = roc_auc_score(y_test, xgb_preds)

# Print results
print(f"Accuracy:  {xgb_accuracy:}")
print(f"Precision: {xgb_precision:}")
print(f"Recall:    {xgb_recall:}")
print(f"F1-score:  {xgb_f1:}")
print(f"ROC-AUC:   {xgb_roc_auc:}")

Accuracy:  0.8076650106458482
Precision: 0.6807017543859649
Recall:    0.5187165775401069
F1-score:  0.5887708649468892
ROC-AUC:   0.7153969361130486


In [24]:
# Optional but helpful
# TODO:
# Print classification report and confusion matrix

print(classification_report(y_test, xgb_preds))
print(confusion_matrix(y_test, xgb_preds)) 

              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.68      0.52      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409

[[944  91]
 [180 194]]


## 13. Feature Importance

Use the trained XGBoost model to:
- extract feature importances,
- recover transformed feature names,
- display the top 5 to 10 most important features.


In [25]:
# TODO:
# Access the fitted preprocessor and fitted XGBoost classifier from the pipeline

# Example structure:
# fitted_preprocessor = best_model.named_steps["preprocessor"]
# fitted_xgb = best_model.named_steps["classifier"]

fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_xgb = final_model.named_steps["classifier"]


In [26]:
# TODO:
# Get transformed feature names
# Get feature importances
# Create a DataFrame sorted by importance

# Example structure:
# feature_names = fitted_preprocessor.get_feature_names_out()
# importances = fitted_xgb.feature_importances_

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_xgb.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)


In [27]:
# TODO:
# Display the top 10 most important features

print(feature_importance_df.head(10))

                                feature  importance
37         cat__Contract_Month-to-month    0.415293
17     cat__InternetService_Fiber optic    0.107321
19               cat__OnlineSecurity_No    0.066205
28                  cat__TechSupport_No    0.058200
44  cat__PaymentMethod_Electronic check    0.036326
0                           num__tenure    0.032705
40             cat__PaperlessBilling_No    0.024530
36             cat__StreamingMovies_Yes    0.024155
39               cat__Contract_Two year    0.022521
38               cat__Contract_One year    0.018899


## 14. Interpret the Top Features

Write a short interpretation of the most important features.

Your explanation should:
- use plain language,
- connect features to churn behavior,
- and explain what the company might learn from them.


**TODO (write your answer below):**

This model truly has 2 majorly important features, Contract-month-to-month and InternetService. Both giving the most information to help make an accurate prediction by spiliting the trees. This is important for many of the features offer any additional information. 

These features connect back to our primary goal of explain patterns of costumers to a company that cannot truly know everything about each costumer who walks in. Predicting behaviors based on just a few key features such as these can prove importance of why costumers are leaving and/or why they want to stay. 

I would tell this company that month-to-month billing cost is the driving feature pattern in costumers leaving the company and the second would be that the lack of information or use of additional comodities like onlinesecurity and tech-support are patterns seen costomers not being satisfied. 

## 15. Final Comparison: Logistic Regression vs XGBoost

Compare the two models using your results.

Your discussion should answer:
- Which model performed better?
- On which metric(s)?
- Why might XGBoost perform better on this dataset?
- What is one limitation of XGBoost?


In [28]:
# TODO:
# Print a side-by-side comparison of the main metrics
# for Logistic Regression and XGBoost

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    " L_regression": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc],
    "XG_boost": [
        xgb_accuracy,
        xgb_precision,
        xgb_recall,
        xgb_f1,
        xgb_roc_auc]
})

print(comparison)

      Metric   L_regression  XG_boost
0   Accuracy       0.805536  0.807665
1  Precision       0.657233  0.680702
2     Recall       0.558824  0.518717
3         F1       0.604046  0.588771
4    ROC-AUC       0.841871  0.715397


## 16. Suggested Submission Checklist

Before submitting, make sure your notebook includes:
- completed code cells,
- outputs for each section,
- your selected main evaluation metric and justification,
- tuned XGBoost hyperparameters,
- feature importance results,
- interpretation of important features,
- and a final comparison of the two models.
